<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook Subpixel Vegetation Line Demonstration

The objective of this notebook is to prepare an annual NDVI value and extract a vegetation line from it at different thresholds.

## Step 1.1: Configure the environment

In [ ]:
import folium
import geopandas as gpd
from pystac.client import Client
from utils import load_data
from utils import scale

# if it can't find this you may need to install
from dea_tools.spatial import subpixel_contours

In [66]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<font color='red'>1.1. Revise the site of training data...


In [67]:
site = "Mejrirok"
filename = f"inputs/{site}-site.geojson"

## Step 1.2: Configure STAC access and search parameters

Do you remember what a stac catalog is...? You can insert this one : https://stac.digitalearthpacific.org


In [68]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

<font color='red'>1.2. Revise the site name

In [69]:
area_of_interest = "..."

# 1. Read the file
aoi = gpd.read_file(f"{filename}")

# 2. Get the bounding box directly from the GeoDataFrame
# This returns exactly what STAC needs: (minx, miny, maxx, maxy)
bbox = aoi.total_bounds

<font color='red'>1.3. Revise the year to `2025` and include the stac catalog collections `dep_s2_geomad`

In [70]:
# 3. Proceed with your search
datetime = '2017/2025'
items = list(
    client.search(
        collections=["dep_s2_geomad"],
        bbox=bbox,
        datetime=datetime
    ).items()
)

#### How many images were found?

In [71]:
print(f"Found {len(items)} items for 2017-2025")

Found 9 items for 2017-2025


In [72]:
measurements = ["nir", "red"]

In [73]:
data = load_data(
    items,
    measurements,
    bbox,
)

In [74]:
data['nir'] = scale(data.nir)
data['red'] = scale(data.red)
data["ndvi"] = (data["nir"] - data["red"]) / (data["nir"] + data["red"])

Try to adjust the value and rerun each time to see how this affects your vegetation mask. Try to find the best value for identifying vegetation. 

In [75]:
ndvi_threshold = 0.3
ndvi_mask = (data.ndvi.squeeze() > ndvi_threshold).astype('uint8')
ndvi_mask.sel(time='2025').odc.explore(vmin=0, vmax=1)

## Extract vegetation contour lines using the threshold above

and save

In [76]:
lines = subpixel_contours(data.ndvi, ndvi_threshold)
# Add year as variable
lines['year'] = lines.time.str[0:4].astype(int)
lines.to_file(f"vegetation_line_{ndvi_threshold}.gpkg")

Loading data into memory using Dask
Operating in single z-value, multiple arrays mode


In [77]:
lines.explore(
    column="year",
    cmap="viridis",
    vmin=2017,
    vmax=2025,
    tiles="Esri.WorldImagery",
)

A smoothed version of the above

In [63]:
# Target pixel size in metres (data is 10 m)
resolution = 5

# Same extent and CRS, smaller pixels, resampled with cubic interpolation
target_geobox = data.ndvi.odc.geobox.zoom_to(resolution=resolution)
ndvi_smoothed = data.ndvi.odc.reproject(target_geobox, resampling="cubic_spline")

In [ ]:
smoothed_lines = subpixel_contours(ndvi_smoothed, ndvi_threshold)
# Add year as variable
smoothed_lines['year'] = smoothed_lines.time.str[0:4].astype(int)
smoothed_lines.to_file(f"vegetation_line_{ndvi_threshold}_smoothed.gpkg")

### Explore EVI

Add some markdown to explain EVI

Input formula to define the index:   
`data["evi"] = (2.5 * data["nir"] - data["red"]) / (data["nir"] + (6 * data["red"]) - (7.5 * data["blue"]) + 1)`

Repeat the steps taken for NDVI for EVI

### Explore SAVI

Add some markdown to explain SAVI

Input formula to define the index:  
`data["savi"] = (data["nir"] - data["red"]) / (data["nir"] + data["red"])`

Repeat the steps taken for NDVI for SAVI

<font color='red'>1.9. Mask out the 'scaled' variable including the ocean 
>`scaled`, mask = all_masks(`scaled`, return_mask = True)


<font color='red'>Explore the extent of this 'mask' variable  
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

<font color='red'>2.5. Export your final `raster` variable to a geojson file.
>`variable`.to_file(f"output/{site}-vector.geo", index=False)

<font color='red'>2.5. Export your final `vector` variable to a geojson file.
>`variable`.to_file(f"output/{site}-vector.geo", index=False)